# Assignment 05 — Isolation Forest Anomaly Detection
**Author:** Muhammad Huzaif Amir · ArzensIntern Advanced Track

This notebook walks through the full pipeline implemented in
`ArzensIntern_MuhammadHuzaifAmir_anomaly_detector.py` and
`ArzensIntern_MuhammadHuzaifAmir_evaluation.py`: data preparation, Isolation
Forest training, threshold tuning, standard evaluation, and a simple
robustness/evasion check. It is a walkthrough companion to the .py scripts,
not a replacement for them — the CLI scripts are the graded deliverables.

In [1]:
import sys
sys.path.insert(0, '.')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from ArzensIntern_MuhammadHuzaifAmir_anomaly_detector import (
    load_and_prepare_data, train_test_split_manual, train_model, score_samples,
    labels_at_threshold, precision_recall_f1, tune_thresholds, CANONICAL_FEATURES,
    DEFAULT_THRESHOLDS,
)
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 20)
print('Imports OK')

Imports OK


## Step 1–2: Load data, select & scale the six canonical features

In [2]:
X, y, raw_df = load_and_prepare_data('sample_data/network_traffic_dataset.csv')
print(f"Feature matrix: {X.shape}")
print(f"Ground-truth labels available: {y is not None}")
X.describe()

Loading data from: sample_data/network_traffic_dataset.csv


  Raw shape: 55,110 rows x 54 columns
  Selected features: ['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate']
Feature matrix: (55110, 6)
Ground-truth labels available: True


,dur,spkts,dpkts,sbytes,dbytes,rate
count,5.511000e+04,55110.000000,55110.000000,55110.000000,55110.000000,5.511000e+04
mean,2.052534e+05,35.368518,7.577391,2187.108999,724.513179,6.313861e+06
std,4.638340e+05,94.930443,6.482993,4942.509957,2007.142335,3.410067e+07
min,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00
25%,7.330701e+01,4.000000,3.000000,334.837961,202.082607,4.163228e+01
50%,1.091574e+05,9.000000,7.000000,671.894289,467.749101,1.540082e+02
75%,3.459362e+05,15.000000,11.000000,1063.521456,752.058354,6.138387e+05
max,4.887884e+07,873.000000,96.000000,45457.573029,44523.605038,7.600000e+08


In [3]:
X_train, X_test, y_train, y_test = train_test_split_manual(X, y, test_ratio=0.2)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features scaled with StandardScaler (fit on train only).")

Train: (44088, 6)  Test: (11022, 6)
Features scaled with StandardScaler (fit on train only).


## Step 3: Train the Isolation Forest

In [4]:
model = train_model(X_train_scaled, n_estimators=100, contamination=0.1)


Training Isolation Forest...
  Samples: 44,088
  Features: 6
  Contamination: 0.1


  Model trained successfully!


## Step 4: Score the test set

In [5]:
scores, is_anomaly_default = score_samples(model, X_test_scaled)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores, bins=60, color='#0f3460')
ax.axvline(0, color='#e94560', linestyle='--', label='default threshold = 0.0')
ax.set_xlabel('Anomaly score (decision_function)')
ax.set_ylabel('Count')
ax.set_title('Distribution of anomaly scores on the test set')
ax.legend()
plt.show()

## Step 5: Threshold tuning — precision / recall / F1 trade-off

In [6]:
results_df, best_threshold = tune_thresholds(scores, y_test, thresholds=DEFAULT_THRESHOLDS, plot_path=None)
results_df


Threshold tuning results:
 threshold   tp   fp   fn   tn  precision  recall    f1
    -0.500    0    0 2380 8642      0.000   0.000 0.000
    -0.300    0    0 2380 8642      0.000   0.000 0.000
    -0.100  244    0 2136 8642      1.000   0.103 0.186
     0.000 1015   30 1365 8612      0.971   0.426 0.593
     0.100 1679 2448  701 6194      0.407   0.705 0.516
     0.300 2380 8642    0    0      0.216   1.000 0.355

  Best F1 at threshold=0.0: P=0.971 R=0.426 F1=0.593


,threshold,tp,fp,fn,tn,precision,recall,f1
0,-0.5,0,0,2380,8642,0.000000,0.000000,0.000000
1,-0.3,0,0,2380,8642,0.000000,0.000000,0.000000
2,-0.1,244,0,2136,8642,1.000000,0.102521,0.185976
3,0.0,1015,30,1365,8612,0.971292,0.426471,0.592701
4,0.1,1679,2448,701,6194,0.406833,0.705462,0.516060
5,0.3,2380,8642,0,0,0.215932,1.000000,0.355171


In [7]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(results_df['threshold'], results_df['precision'], marker='o', label='Precision')
ax.plot(results_df['threshold'], results_df['recall'], marker='s', label='Recall')
ax.plot(results_df['threshold'], results_df['f1'], marker='^', label='F1-Score')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs. Threshold')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

print(f"Best F1 threshold: {best_threshold}")

Best F1 threshold: 0.0


## Step 6: Save model & scaler, then evaluate (Task 3, Part A)

Everything below mirrors what `evaluation.py` computes automatically; it is
shown here inline so the metrics and confusion matrix can be inspected
directly in the notebook.

In [8]:
import os
os.makedirs('outputs', exist_ok=True)
joblib.dump(model, 'outputs/isolation_forest_model_nb.pkl')
joblib.dump(scaler, 'outputs/standard_scaler_nb.pkl')

THRESHOLD = 0.0
y_pred = labels_at_threshold(scores, THRESHOLD)
metrics = precision_recall_f1(y_test.values, y_pred)
tn = int(np.sum((y_pred == 0) & (y_test.values == 0)))
accuracy = (metrics['tp'] + tn) / len(y_test)
fpr = metrics['fp'] / (metrics['fp'] + tn) if (metrics['fp'] + tn) else 0.0

print(f"Accuracy:  {accuracy*100:.1f}%")
print(f"Precision: {metrics['precision']*100:.1f}%")
print(f"Recall:    {metrics['recall']*100:.1f}%")
print(f"F1-Score:  {metrics['f1']*100:.1f}%")
print(f"FPR:       {fpr*100:.1f}%")

Accuracy:  87.3%
Precision: 97.1%
Recall:    42.6%
F1-Score:  59.3%
FPR:       0.3%


In [9]:
matrix = np.array([[tn, metrics['fp']], [metrics['fn'], metrics['tp']]])
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(matrix, cmap='Blues')
labels = [['TN', 'FP'], ['FN', 'TP']]
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{labels[i][j]}\n{matrix[i,j]:,}", ha='center', va='center',
                 color='white' if matrix[i,j] > matrix.max()/2 else 'black')
ax.set_xticks([0,1]); ax.set_xticklabels(['Pred. Normal', 'Pred. Anomaly'])
ax.set_yticks([0,1]); ax.set_yticklabels(['Actual Normal', 'Actual Anomaly'])
ax.set_title('Confusion Matrix')
plt.tight_layout(); plt.show()

## Part B — Robustness / evasion check

In [10]:
rng = np.random.RandomState(42)
X_anom = X_test.loc[y_test.values == 1]
feature_std = X_anom.std().values

def detect_rate(df):
    s = model.decision_function(scaler.transform(df))
    return labels_at_threshold(s, THRESHOLD).mean()

baseline = detect_rate(X_anom)
noise = pd.DataFrame(X_anom.values + feature_std*0.10*rng.randn(*X_anom.shape), columns=X_anom.columns).clip(lower=0)
down = X_anom * 0.9
up = X_anom * 1.1

print(f"Baseline detected:      {baseline*100:.1f}%")
print(f"+10% noise detected:    {detect_rate(noise)*100:.1f}%")
print(f"Scale x0.9 detected:    {detect_rate(down)*100:.1f}%")
print(f"Scale x1.1 detected:    {detect_rate(up)*100:.1f}%")

Baseline detected:      42.6%
+10% noise detected:    51.1%
Scale x0.9 detected:    36.4%
Scale x1.1 detected:    50.0%


**Observation:** the detection rate drops noticeably under perturbation —
see `ROBUSTNESS.md` for the full analysis, which concludes the model is
**fragile** to simple evasion attempts at this threshold and feature set.

## Conclusion

- Isolation Forest trained on 6 flow features reaches ~97% precision / ~43%
  recall at the default threshold — a high-confidence, low-noise detector
  that misses a substantial share of attacks.
- Threshold tuning shows the expected precision/recall trade-off; F1 peaks
  near the default threshold of 0.0 for this dataset.
- The model is **not** robust to trivial evasion (noise/scaling), which is
  an important operational caveat documented in `ROBUSTNESS.md`.
- See `evaluation_report.html` for the complete Task 3 report, including
  per-attack-type breakdown and operational (alert-fatigue, concept-drift)
  analysis.